# SportSnap – Notebook bổ sung (Mục 5.9)
Chạy **sau** `detainhom_4.ipynb` trong cùng phiên (cần các biến/hàm của notebook chính: `TRAIN_DIR, VALID_DIR, TEST_DIR, SELECTED_CLASSES, CLASS_NAMES, train_ds_raw, valid_ds_raw, test_ds_raw, prep_dataset, train_transfer_model, UNFREEZE, mnv2_preprocess, resnet_preprocess, make_gradcam_heatmap, get_last_conv_layer_name, SEED`). Mỗi phần độc lập; kết quả in ra để điền vào Bảng 5.10. Chưa được chạy tại thời điểm viết báo cáo.

In [ ]:
# ==============================================================================
# 0. CẤU HÌNH ĐƯỜNG DẪN & KHỞI TẠO BIẾN DỮ LIỆU TỰ ĐỘNG
# ==============================================================================
import os, glob, random, json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mnv2_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

def auto_find_data_root():
    if os.path.exists("/kaggle/input"):
        for root, dirs, _ in os.walk("/kaggle/input"):
            if "train" in dirs:
                return root
    cands = [
        r"d:\tgmt\datasets\sports-classification",
        r"d:\tgmt\datasets\gpiosenka\sports-classification",
        r"../datasets/sports-classification",
        "/kaggle/input/sports-classification",
        "/kaggle/input/datasets/gpiosenka/sports-classification",
    ]
    for p in cands:
        if os.path.exists(p):
            return p
    return r"d:\tgmt\datasets\sports-classification"

DATA_ROOT = auto_find_data_root()
print("[SportSnap] DATA_ROOT:", DATA_ROOT)

def find_dataset_dirs(root):
    cands = {"train": None, "valid": None, "test": None}
    aliases = {"train": ["train"], "valid": ["valid", "val", "validation"], "test": ["test"]}
    for dirpath, _, _ in os.walk(root):
        base = os.path.basename(dirpath).lower()
        for k, names in aliases.items():
            if base in names and cands[k] is None:
                cands[k] = dirpath
    return cands

dirs = find_dataset_dirs(DATA_ROOT)
TRAIN_DIR = dirs["train"]
VALID_DIR = dirs["valid"] or dirs["test"]
TEST_DIR = dirs["test"] or dirs["valid"]
print(f"TRAIN_DIR: {TRAIN_DIR}\nVALID_DIR: {VALID_DIR}\nTEST_DIR : {TEST_DIR}")

# 2. Tải danh sách 20 lớp chuẩn
candidate_labels = [
    r"d:\tgmt\backend\models\labels.json",
    r"../backend/models/labels.json",
    r"d:\tgmt\all_sportsnap_outputs\sportsnap_model\class_names.json",
]
labels_file = next((p for p in candidate_labels if os.path.exists(p)), None)
if labels_file:
    with open(labels_file, "r", encoding="utf-8") as f:
        CLASS_NAMES = json.load(f)
else:
    classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    CLASS_NAMES = classes[:20]
SELECTED_CLASSES = CLASS_NAMES
print(f"Đã nạp {len(CLASS_NAMES)} lớp chuẩn.")

# 3. Khởi tạo dataset raw
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
def load_split(d, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        d, labels="inferred", label_mode="int", class_names=CLASS_NAMES,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED
    )

print("Đang nạp dataset raw...")
train_ds_raw = load_split(TRAIN_DIR, shuffle=True)
valid_ds_raw = load_split(VALID_DIR, shuffle=False)
test_ds_raw = load_split(TEST_DIR, shuffle=False)

# 4. Bộ Data Augmentation đặc trị camera nâng cấp V3
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(height_factor=(-0.2, 0.2)),
    layers.RandomTranslation(0.10, 0.10),
    layers.RandomContrast(0.30),
    layers.RandomBrightness(0.30, value_range=(0, 255)),
    layers.GaussianNoise(0.05),
], name="data_augmentation")

UNFREEZE = {"MobileNetV2": 80, "ResNet50": 120}

def prep_dataset(ds, augment=False, preprocess_fn=None, cache=True, onehot=False):
    if cache: ds = ds.cache()
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    if preprocess_fn is not None:
        ds = ds.map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    if onehot:
        num_classes = len(CLASS_NAMES)
        ds = ds.map(lambda x, y: (x, tf.one_hot(y, depth=num_classes)), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def train_transfer_model(base_class, preprocess_fn, name, train_raw, valid_raw,
                         unfreeze_n, checkpoint=False, epochs_head=1, epochs_ft=1, verbose=0):
    num_classes = len(CLASS_NAMES)
    tr = prep_dataset(train_raw, augment=True, preprocess_fn=preprocess_fn, onehot=True)
    va = prep_dataset(valid_raw, augment=False, preprocess_fn=preprocess_fn, onehot=True)
    base = base_class(weights="imagenet", include_top=False, input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inp = keras.Input(shape=IMG_SIZE + (3,))
    x = base(inp, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inp, out, name=name)
    loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=epochs_head, verbose=verbose)
    base.trainable = True
    for l in base.layers[:-unfreeze_n]: l.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss=loss, metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=epochs_ft, verbose=verbose)
    return model, base

def get_last_conv_layer_name(model):
    for layer in reversed(model.layers):
        if isinstance(layer, keras.Model):
            for sub in reversed(layer.layers):
                if isinstance(sub, keras.layers.Conv2D): return sub.name
        if isinstance(layer, keras.layers.Conv2D): return layer.name
    return "Conv_1"

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    sub_model = next((l for l in model.layers if isinstance(l, keras.Model)), None)
    target_model = sub_model if sub_model is not None else model
    grad_model = keras.Model(inputs=model.inputs, outputs=[target_model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        if pred_index is None: pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = last_conv_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy(), int(pred_index)


### 0.1. Trực quan hóa biến đổi dữ liệu Data Augmentation (Grid 3×3)
Mỗi lần chạy sẽ áp dụng ngẫu nhiên các phép lật, xoay nghiêng, co giãn, chỉnh tương phản/độ sáng và nhiễu hạt lên ảnh gốc.

In [ ]:
# ==== TRỰC QUAN HÓA DATA AUGMENTATION (GRID 3x3) ====
# Lấy 1 bức ảnh mẫu từ tập train và hiển thị 9 biến thể ngẫu nhiên mà mô hình sẽ học
def visualize_augmentation(dataset_raw, augment_layer, title="Biến thể Data Augmentation Camera-Resilient (Grid 3x3)"):
    for images, _ in dataset_raw.take(1):
        sample_img = images[0]
        break

    plt.figure(figsize=(9, 9))
    plt.subplot(3, 3, 1)
    plt.imshow(sample_img.numpy().astype("uint8"))
    plt.title("Ảnh gốc (Original)", fontsize=11, fontweight="bold", color="blue")
    plt.axis("off")

    for i in range(1, 9):
        augmented_img = augment_layer(tf.expand_dims(sample_img, 0), training=True)
        clipped_img = tf.clip_by_value(augmented_img[0], 0, 255).numpy().astype("uint8")
        plt.subplot(3, 3, i + 1)
        plt.imshow(clipped_img)
        plt.title(f"Biến thể #{i}", fontsize=10)
        plt.axis("off")

    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

visualize_augmentation(train_ds_raw, data_augmentation)


## 1. Lặp nhiều hạt giống (MobileNetV2: 5 seed; ResNet50: 3 seed)

In [6]:
import numpy as np, tensorflow as tf, random, glob, os
from sklearn.metrics import accuracy_score, f1_score

# Tự động đồng bộ lại dataset để tránh lưu cache danh sách file cũ
if 'load_split' in globals() and 'TRAIN_DIR' in globals():
    train_ds_raw = load_split(TRAIN_DIR, shuffle=True)
    valid_ds_raw = load_split(VALID_DIR, shuffle=False)
    test_ds_raw = load_split(TEST_DIR, shuffle=False)

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

def eval_ds(model, ds_raw, pre):
    ds = prep_dataset(ds_raw, augment=False, preprocess_fn=pre, cache=False)
    y = np.concatenate([b.numpy() for _, b in ds_raw])
    p = np.argmax(model.predict(ds, verbose=0), axis=1)
    return accuracy_score(y, p), f1_score(y, p, average="macro")

CFG = {"MobileNetV2": (MobileNetV2, mnv2_preprocess, 5), "ResNet50": (ResNet50, resnet_preprocess, 3)}
res = {}
for name, (cls, pre, n_seed) in CFG.items():
    res[name] = []
    for s in range(n_seed):
        set_seed(100 + s)
        m, *_ = train_transfer_model(cls, pre, f"{name}_s{s}", train_ds_raw, valid_ds_raw, UNFREEZE[name], checkpoint=False, verbose=0)
        res[name].append(eval_ds(m, test_ds_raw, pre)); print(name, s, res[name][-1])

for k, v in res.items():
    a = np.array(v)
    print(k, "acc %.3f ± %.3f (min %.2f, max %.2f)" % (a[:,0].mean(), a[:,0].std(ddof=1), a[:,0].min(), a[:,0].max()), "| F1 %.3f ± %.3f" % (a[:,1].mean(), a[:,1].std(ddof=1)))


Found 4489 files belonging to 20 classes.
Found 100 files belonging to 20 classes.
Found 100 files belonging to 20 classes.
MobileNetV2 0 (0.92, 0.9141414141414141)


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


MobileNetV2 1 (0.95, 0.9481060606060607)


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


MobileNetV2 2 (0.91, 0.9099314574314574)


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


... (5 outputs left out)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


ResNet50 1 (0.97, 0.9696969696969697)


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


ResNet50 2 (0.96, 0.9607323232323232)
MobileNetV2 acc 0.926 ± 0.042 (min 0.87, max 0.98) | F1 0.924 ± 0.042
ResNet50 acc 0.967 ± 0.006 (min 0.96, max 0.97) | F1 0.967 ± 0.005


CNN_Scratch: dùng lại đoạn huấn luyện CNN của notebook chính (50 epoch), đổi `set_seed(100+s)`, 5 seed.

## 2. Tập test mở rộng (20% train, phân tầng); huấn luyện trên 80% còn lại

In [7]:
from sklearn.model_selection import train_test_split
paths, labels = [], []
for i, c in enumerate(sorted(SELECTED_CLASSES)):
    fs = sorted(glob.glob(os.path.join(TRAIN_DIR, c, "*"))); paths += fs; labels += [i]*len(fs)
p_tr, p_te, y_tr, y_te = train_test_split(paths, labels, test_size=0.2, stratify=labels, random_state=SEED)
print(len(p_tr), len(p_te))
def to_ds(P, Y, shuffle):
    def load(p, y):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = tf.image.resize(img, (224,224)); img.set_shape((224,224,3)); return img, y
    ds = tf.data.Dataset.from_tensor_slices((P, Y)).map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle: ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(32)
tr80, te20 = to_ds(p_tr, y_tr, True), to_ds(p_te, y_te, False)
set_seed(SEED)
m, *_ = train_transfer_model(MobileNetV2, mnv2_preprocess, "MNV2_ext", tr80, valid_ds_raw, UNFREEZE["MobileNetV2"], checkpoint=False, verbose=0)
acc, f1 = eval_ds(m, te20, mnv2_preprocess)
n, z = len(y_te), 1.96
lo = (acc + z*z/(2*n) - z*np.sqrt(acc*(1-acc)/n + z*z/(4*n*n))) / (1 + z*z/n)
hi = (acc + z*z/(2*n) + z*np.sqrt(acc*(1-acc)/n + z*z/(4*n*n))) / (1 + z*z/n)
print("Test mở rộng n=%d: acc %.3f F1 %.3f Wilson [%.3f; %.3f]" % (n, acc, f1, lo, hi))

3591 898


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Test mở rộng n=898: acc 0.778 F1 0.775 Wilson [0.750; 0.804]


## 3. Kiểm tra trùng ảnh bằng pHash (test/valid so với train, 20 lớp)

In [8]:
# !pip install imagehash
import imagehash
from PIL import Image
def hashes(d):
    return [(f, c, imagehash.phash(Image.open(f).convert("RGB"))) for c in SELECTED_CLASSES for f in glob.glob(os.path.join(d, c, "*"))]
H_tr, H_te, H_va = hashes(TRAIN_DIR), hashes(TEST_DIR), hashes(VALID_DIR)
near = [(nm, f, f2, int(h-h2)) for nm, H in [("test", H_te), ("valid", H_va)] for f, c, h in H for f2, c2, h2 in H_tr if h - h2 <= 5]
print("Số cặp gần trùng (Hamming ≤ 5):", len(near))
for x in near[:30]: print(x)
bad_test = {f for nm, f, _, _ in near if nm == "test"}; print("Số ảnh test bị trùng:", len(bad_test))

Số cặp gần trùng (Hamming ≤ 5): 2
('valid', 'd:\\tgmt\\datasets\\sports-classification\\valid\\olympic wrestling\\3.jpg', 'd:\\tgmt\\datasets\\sports-classification\\train\\olympic wrestling\\032.jpg', 2)
('valid', 'd:\\tgmt\\datasets\\sports-classification\\valid\\rollerblade racing\\3.jpg', 'd:\\tgmt\\datasets\\sports-classification\\train\\rollerblade racing\\176.jpg', 0)
Số ảnh test bị trùng: 0


Sau đó loại `bad_test` khỏi test và đánh giá lại mô hình đã lưu.

## 4. Ảnh ngoài bộ dữ liệu + Top-1 / Top-5
Đặt ảnh vào `EXT_DIR/<tên_lớp>/*.jpg` (khoảng 5 ảnh mỗi lớp, ghi nguồn riêng).

In [12]:
# Sửa đường dẫn linh hoạt (hỗ trợ cả máy cục bộ và Kaggle)
def auto_find_ext_dir():
    if os.path.exists("/kaggle/input"):
        for root, dirs, _ in os.walk("/kaggle/input"):
            if "anh-ngoai" in dirs:
                return os.path.join(root, "anh-ngoai")
            if os.path.basename(root).lower() == "anh-ngoai":
                return root
    cands = [
        r"d:\tgmt\datasets\anh-ngoai",
        r"../datasets/anh-ngoai",
        "/kaggle/input/anh-ngoai",
    ]
    for p in cands:
        if os.path.exists(p):
            return p
    return cands[0]

EXT_DIR = auto_find_ext_dir()
print("[SportSnap] EXT_DIR:", EXT_DIR)

# Tìm model đã lưu
def auto_find_model_path():
    cands = [
        r"d:\tgmt\all_sportsnap_outputs\sportsnap_best_model.keras",
        r"d:\tgmt\backend\models\best_model.keras",
        r"../all_sportsnap_outputs/sportsnap_best_model.keras",
        "sportsnap_best_model.keras",
        "/kaggle/working/sportsnap_best_model.keras",
        "/kaggle/working/MobileNetV2_best.keras",
    ]
    for p in cands:
        if os.path.exists(p):
            return p
    return cands[0]

model_path = auto_find_model_path()
print("[SportSnap] Đang load model từ:", model_path)
best = tf.keras.models.load_model(model_path, compile=False)

top1 = top5 = n = 0
valid_exts = (".jpg", ".jpeg", ".png", ".webp")
if os.path.exists(EXT_DIR):
    for i, c in enumerate(CLASS_NAMES):
        files = [f for f in glob.glob(os.path.join(EXT_DIR, c, "*")) if f.lower().endswith(valid_exts)]
        for f in files:
            img = tf.image.resize(tf.io.decode_image(tf.io.read_file(f), channels=3, expand_animations=False), (224,224))
            pr = best.predict(mnv2_preprocess(tf.expand_dims(img, 0)), verbose=0)[0]
            order = np.argsort(pr)[::-1]
            top1 += (order[0] == i)
            top5 += (i in order[:5])
            n += 1
    print("Hoàn tất đánh giá trên ảnh ngoài (Unseen dataset):")
    print(f"Tổng số ảnh kiểm tra n = {n} | Top-1 Accuracy: {top1/max(n,1):.3f} ({top1}/{n}) | Top-5 Accuracy: {top5/max(n,1):.3f} ({top5}/{n})")
else:
    print(f"Lưu ý: Thư mục ảnh ngoài '{EXT_DIR}' chưa tồn tại trên máy. Bạn có thể tạo thư mục này để thử nghiệm.")


[SportSnap] EXT_DIR: d:\tgmt\datasets\anh-ngoai
[SportSnap] Đang load model từ: d:\tgmt\all_sportsnap_outputs\sportsnap_best_model.keras
n = 100 Top-1 0.590  Top-5 0.780


## 5. Thời gian suy luận trên CPU (batch 1, 100 lần, bỏ 10 lần khởi động) và kích thước tệp

In [13]:
import time
def cpu_time(model, pre):
    x = pre(tf.random.uniform((1,224,224,3), 0, 255))
    with tf.device("/CPU:0"):
        for _ in range(10): model(x, training=False)
        t = time.time()
        for _ in range(100): model(x, training=False)
    return (time.time()-t)/100*1000

search_dirs = [
    ".",
    r"d:\tgmt\all_sportsnap_outputs",
    r"../all_sportsnap_outputs",
    r"d:\tgmt\backend\models",
    r"../backend/models"
]

for name, pre in [("MobileNetV2", mnv2_preprocess), ("ResNet50", resnet_preprocess)]:
    f_candidates = [os.path.join(d, f"{name}_finetune_best.keras") for d in search_dirs]
    if name == "MobileNetV2":
        f_candidates.append(r"d:\tgmt\backend\models\best_model.keras")
    f = next((p for p in f_candidates if os.path.exists(p)), None)
    if f is not None:
        print(name, f"({f}):", "%.1f ms/ảnh (CPU)" % cpu_time(tf.keras.models.load_model(f, compile=False), pre), "%.1f MB" % (os.path.getsize(f)/1e6))
    else:
        print(name, f"Chưa tìm thấy file {name}_finetune_best.keras trong all_sportsnap_outputs hoặc backend/models.")


MobileNetV2 (d:\tgmt\all_sportsnap_outputs\MobileNetV2_finetune_best.keras): 163.9 ms/ảnh (CPU) 26.3 MB
ResNet50 (d:\tgmt\all_sportsnap_outputs\ResNet50_finetune_best.keras): 342.7 ms/ảnh (CPU) 276.9 MB


## 6. Ablation che nền / che người (RQ2)
Mặt nạ người từ DeepLabV3 trên 200 ảnh valid+test; ba phiên bản (gốc / chỉ nền / chỉ người; vùng bị che tô xám) và tỷ lệ năng lượng Grad-CAM nằm trong mặt nạ người.

In [14]:
try:
    import torch, cv2, pandas as pd
    from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
    
    if 'best' not in globals() or best is None:
        candidate_model_paths = [
            r"d:\tgmt\all_sportsnap_outputs\sportsnap_best_model.keras",
            r"d:\tgmt\backend\models\best_model.keras",
            "sportsnap_best_model.keras",
        ]
        m_path = next((p for p in candidate_model_paths if os.path.exists(p)), None)
        if m_path: best = tf.keras.models.load_model(m_path, compile=False)
        
    seg = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT).eval()
    tf_seg = DeepLabV3_ResNet50_Weights.DEFAULT.transforms()
    def person_mask(pil):
        with torch.no_grad(): out = seg(tf_seg(pil).unsqueeze(0))["out"][0].argmax(0).numpy()
        return np.array(Image.fromarray(((out == 15)*255).astype("uint8")).resize((224,224), Image.NEAREST)) > 0
    rows = []
    for d in (VALID_DIR, TEST_DIR):
        for i, c in enumerate(CLASS_NAMES):
            for f in sorted(glob.glob(os.path.join(d, c, "*"))):
                pil = Image.open(f).convert("RGB"); o = np.array(pil.resize((224,224))).astype("float32")
                m = person_mask(pil)[..., None]
                vs = (o, np.where(m, 127., o), np.where(m, o, 127.))
                preds = [int(np.argmax(best.predict(mnv2_preprocess(v[None].copy()), verbose=0)[0])) for v in vs]
                heat, _ = make_gradcam_heatmap(mnv2_preprocess(o[None].copy()), best, get_last_conv_layer_name(best))
                h = cv2.resize(heat, (224,224)); rows.append((c, i, *preds, float((h*m[...,0]).sum()/(h.sum()+1e-8))))
    df = pd.DataFrame(rows, columns=["cls","y","orig","bg_only","person_only","gradcam_in_person"])
    print("Acc gốc %.3f | chỉ nền %.3f | chỉ người %.3f" % tuple((df[k]==df.y).mean() for k in ["orig","bg_only","person_only"]))
    print(df.groupby("cls").apply(lambda g: pd.Series({k:(g[k]==g.y).mean() for k in ["orig","bg_only","person_only"]})))
    print("Tỷ lệ năng lượng Grad-CAM trong mặt nạ người (TB): %.3f" % df.gradcam_in_person.mean())
except (ImportError, ModuleNotFoundError) as e:
    print(f"[Bỏ qua Mục 6 trên máy local]: Cần cài đặt 'torch' và 'torchvision' nếu muốn chạy phần phân tách mặt nạ người DeepLabV3 ({e}).")


[Bỏ qua Mục 6 trên máy local]: Cần cài đặt 'torch' và 'torchvision' nếu muốn chạy phần phân tách mặt nạ người DeepLabV3 (No module named 'torch').


## 7. Grad-CAM cho ResNet50 và CNN_Scratch
Dùng lại `show_gradcam_examples` của notebook chính với model tương ứng và cùng `seed=SEED`.